# Scenario: Uncovering Hidden Display Lag

In [6]:
import pandas as pd
import sqlite3
# creating dataset capturing timestamps in seconds (Unix Epoch style or standard strings)
ui_latency_data = {
    "alert_id": [601, 602, 603, 604, 605],
    "patient_id": ["P-801", "P-802", "P-803", "P-804", "P-805"],
    "ai_computed_timestamp": ["2026-06-04 10:00:00", "2026-06-04 10:05:00", "2026-06-04 10:12:00", "2026-06-04 10:20:00", "2026-06-04 10:35:00"],
    "ui_displayed_timestamp": ["2026-06-04 10:00:02", "2026-06-04 10:05:45", "2026-06-04 10:12:01", "2026-06-04 10:24:30", "2026-06-04 10:35:03"],
    # Look at 602 and 604! 45 seconds and 4.5 minutes of lag!
}
# adding the dataset to DataFrame
df_ui_latency_data = pd.DataFrame(ui_latency_data)
# creating sql and save dataframe to save the dataframe in temp memory 
connt = sqlite3.connect(":memory:")
df_ui_latency_data.to_sql("ui_logs", connt, index = False, if_exists = "replace")
print("************************** UI Latency Audit Database is ready! *******************")
# function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)

************************** UI Latency Audit Database is ready! *******************


# Isolating Interface Lag (in Seconds)

In [10]:
# SQL query for all data to review
all_data = "SELECT * FROM ui_logs"
display(run_query(all_data))
print()
# SQL query that calculates the exact delay in seconds between ai_computed_timestamp and ui_displayed_timestamp for every alert.
time_difference = """
SELECT
    patient_id,
    ai_computed_timestamp,
    ui_displayed_timestamp,
    (julianday(ui_displayed_timestamp) - julianday(ai_computed_timestamp)) * 86400 AS display_lag_seconds
FROM ui_logs   
"""
print("***************************** time difference between ai_computed_timestamp and ui_displayed_timestamp ************")
display(run_query(time_difference))

,alert_id,patient_id,ai_computed_timestamp,ui_displayed_timestamp
0,601,P-801,2026-06-04 10:00:00,2026-06-04 10:00:02
1,602,P-802,2026-06-04 10:05:00,2026-06-04 10:05:45
2,603,P-803,2026-06-04 10:12:00,2026-06-04 10:12:01
3,604,P-804,2026-06-04 10:20:00,2026-06-04 10:24:30
4,605,P-805,2026-06-04 10:35:00,2026-06-04 10:35:03



***************************** time difference between ai_computed_timestamp and ui_displayed_timestamp ************


,patient_id,ai_computed_timestamp,ui_displayed_timestamp,display_lag_seconds
0,P-801,2026-06-04 10:00:00,2026-06-04 10:00:02,2.000029
1,P-802,2026-06-04 10:05:00,2026-06-04 10:05:45,44.999997
2,P-803,2026-06-04 10:12:00,2026-06-04 10:12:01,1.000035
3,P-804,2026-06-04 10:20:00,2026-06-04 10:24:30,270.000024
4,P-805,2026-06-04 10:35:00,2026-06-04 10:35:03,3.000024


# Identifying the Danger Zone

In [11]:
# SQL query to filter and display only the alerts where the display lag was greater than 10 seconds
danger_zone = """
SELECT
    patient_id,
    ai_computed_timestamp,
    ui_displayed_timestamp,
    (julianday(ui_displayed_timestamp) - julianday(ai_computed_timestamp)) * 86400 AS display_lag_seconds
FROM ui_logs 
WHERE display_lag_seconds > 10
"""
print("********************************* The Danger Zone *****************")
display(run_query(danger_zone))

********************************* The Danger Zone *****************


,patient_id,ai_computed_timestamp,ui_displayed_timestamp,display_lag_seconds
0,P-802,2026-06-04 10:05:00,2026-06-04 10:05:45,44.999997
1,P-804,2026-06-04 10:20:00,2026-06-04 10:24:30,270.000024
